
## Build Drivers Dimension
1. Read silver drivers table
2. Read gold ref_nationality_region table
3. Join the data from drivers with ref_nationality_region using nationality
4. Select the required columns
5. drivers.driver_id
6. drivers.driver_name
7. drivers.date_of_birth
8. drivers.nationality
9. ref_nationality_region.region
10.Write the transformed data to gold dim_drivers table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00_common/01_environment_config

In [0]:
%run "../00_common/04_gold_helpers"

In [0]:
from pyspark.sql import functions as f

## Step 1 - Read source tables
- silver.drivers
- gold.ref_nationality_region

In [0]:
drivers_path = f"{s3_root_path}/Silver/Kalashri_increment/drivers"
nationality_region_gold_path = "s3://bizf1project123/Gold/Kalashri_increment/ref_nationality_region"

target_path = f"{s3_root_path}/Gold/Kalashri_increment/dim_drivers"

In [0]:
drivers_df = spark.read.format("delta").load(drivers_path)
nationality_region_df = spark.read.format("delta").load(nationality_region_gold_path)

display(drivers_df)
display(nationality_region_df)

## Step 2 - Join drivers with nationality_region_df using nationality
Select the following columns

1. drivers.driver_id
2. drivers.driver_name
3. drivers.date_of_birth
4. drivers.nationality
5. ref_nationality_region.region

In [0]:
dim_drivers_df = (
    drivers_df
        .join(
            nationality_region_df,
            drivers_df.nationality == nationality_region_df.nationality,
            "left"
        )
        .select(
            drivers_df.driver_id,
            drivers_df.driver_name,
            drivers_df.date_of_birth,
            drivers_df.nationality,
            nationality_region_df.region.alias("nationality_region")
        )
)

display(dim_drivers_df)

## Step 3 - Write the transformed data to the gold dim_drivers table

In [0]:
write_to_gold(
    input_df=dim_drivers_df,
    target_path=target_path,
    merge_condition="t.driver_id = s.driver_id",
    columns_to_update=[
        "driver_name",
        "date_of_birth",
        "nationality",
        "nationality_region"
    ]
)

In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Gold/Kalashri_increment/dim_drivers`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Gold/Kalashri_increment/dim_drivers",
#     True
# )